# Air Quality Patterns in African Cities
### Data Analytics Capstone — Group 5

**Client:** United Nations Environment Programme (UNEP)
**Team:** Samuel Tokoye, Paul Kibet Miningwa, Winnie Odoyo, Gloria Simiyu Wandabwa
**Data source:** OpenAQ v3 API
**Cities studied:** Nairobi, Kampala, Kigali, Addis Ababa, Johannesburg, Lagos
**Study period:** 2021–present

## Project brief

Air pollution is an increasing public-health concern across African cities. UNEP has commissioned
this analysis to evaluate air-quality patterns across six selected cities, in order to guide
future intervention priorities.

## Research questions

1. Which city recorded the highest average PM2.5 concentration during the study period?
2. Which pollutants are monitored most frequently across the selected cities?
3. Which cities experience the greatest seasonal variation in PM2.5 concentrations?
4. How frequently do PM2.5 measurements exceed the WHO recommended guideline values?
5. Which monitoring stations have the most complete and reliable datasets?
6. Which cities should be prioritised for air-quality intervention programmes, and why?

Every recommendation in this notebook is directly supported by evidence obtained during the
analysis below.


## 1. Data Acquisition: API Documentation

**API:** OpenAQ v3 (`https://api.openaq.org/v3`)
**Authentication:** API key required, sent via the `X-API-Key` header

### Endpoints used

**`GET /v3/locations`** — identify monitoring stations within each city.

| Parameter | Value used | Notes |
|---|---|---|
| `coordinates` | `{lat},{lon}` per city center | e.g. Nairobi: `-1.286389,36.817223` |
| `radius` | `25000` (metres) | 25km — the maximum radius v3 allows |
| `limit` | `100` | Sufficient for all cities studied |

Each location response includes an embedded `sensors` array (pollutant type, sensor ID) and
`datetimeFirst`/`datetimeLast` reporting window, so a separate `/sensors` call was unnecessary.

**`GET /v3/sensors/{sensor_id}/days`** — pull daily-averaged PM2.5 measurements for each active sensor.

| Parameter | Value used | Notes |
|---|---|---|
| `date_from` | `2021-01-01` | Start of study window |
| `date_to` | current date | End of study window |
| `limit` | `1000` | Max page size |
| `page` | incremented until a page returns <1000 results | `meta.found` returned inconsistent formats across responses, so page-size was used as the reliable pagination stop condition instead |

### Preprocessing rules applied during acquisition

- **Pollutant scope:** full time-series pulled for **PM2.5 only**. Other pollutants were catalogued
  by station frequency (for Q2) but not pulled as time-series — none of the six client questions
  require trend data for pollutants other than PM2.5.
- **"Active sensor" definition:** a sensor counts as active if its station's `datetimeLast` falls
  within 2 years of the pull date. Only active sensors had measurement history pulled.
- **Known limitation — Johannesburg:** 7 PM2.5 sensors found, only 1 active. Johannesburg's overall
  monitoring is also skewed toward SO2/PM10/CO rather than PM2.5, likely reflecting
  industrial/regulatory monitoring infrastructure rather than the low-cost sensor networks
  driving PM2.5 density in Kampala, Lagos, and Nairobi. Retained in the study with this limitation
  documented rather than dropped.

### Files produced by acquisition (see `scripts/`)

| File | Contents |
|---|---|
| `data/raw/{city}_locations.json` | Raw station metadata per city |
| `data/raw/pm25_sensors_summary.json` | Extracted PM2.5 sensors, active/inactive flag |
| `data/raw/{city}_measurements.json` | Daily-averaged PM2.5 readings per active sensor |
| `data/raw/pollutant_frequency.csv` | Station counts per pollutant type per city |


## 2. Data Loading

Load the outputs of the acquisition scripts (see `scripts/`) so the rest of this notebook works from saved data, not live API calls.

In [12]:
import json
import os
import pandas as pd

CITIES = ["nairobi", "kampala", "kigali", "addis_ababa", "johannesburg", "lagos"]
DATA_DIR = "../data/raw"


### 2.1 Station metadata per city

In [13]:
locations = {}
for city in CITIES:
    with open(f"{DATA_DIR}/{city}_locations.json") as f:
        locations[city] = json.load(f)
    print(f"{city}: {locations[city]['meta']['found']} stations found")


nairobi: 16 stations found
kampala: 36 stations found
kigali: 4 stations found
addis_ababa: 9 stations found
johannesburg: 12 stations found
lagos: 61 stations found


### 2.2 PM2.5 sensor summary (active/inactive flag)

In [14]:
with open(f"{DATA_DIR}/pm25_sensors_summary.json") as f:
    sensors_df = pd.DataFrame(json.load(f))

print(f"Total PM2.5 sensors: {len(sensors_df)}")
print(f"Active sensors: {sensors_df['is_active'].sum()}")
sensors_df.groupby("city")["is_active"].agg(["sum", "count"]).rename(
    columns={"sum": "active", "count": "total"}
)


Total PM2.5 sensors: 133
Active sensors: 119


,active,total
city,,
addis_ababa,7,9
johannesburg,1,7
kampala,35,36
kigali,3,4
lagos,61,61
nairobi,12,16


### 2.3 Daily PM2.5 measurements per city

In [15]:
measurements = {}
for city in CITIES:
    path = f"{DATA_DIR}/{city}_measurements.json"
    if os.path.exists(path):
        with open(path) as f:
            measurements[city] = json.load(f)
        total_days = sum(len(s["daily_averages"]) for s in measurements[city])
        print(f"{city}: {len(measurements[city])} active sensor(s), {total_days} total daily records")
    else:
        print(f"{city}: no measurements file found")


nairobi: 12 active sensor(s), 2108 total daily records
kampala: 35 active sensor(s), 7271 total daily records
kigali: 3 active sensor(s), 1418 total daily records
addis_ababa: 7 active sensor(s), 2851 total daily records
johannesburg: 1 active sensor(s), 3 total daily records
lagos: 61 active sensor(s), 9064 total daily records


### 2.4 Pollutant monitoring frequency (Q2 data source)

In [16]:
pollutant_freq = pd.read_csv(f"{DATA_DIR}/pollutant_frequency.csv")
pollutant_freq[pollutant_freq["station_count"] > 0].sort_values(
    ["city", "station_count"], ascending=[True, False]
)


,city,pollutant,station_count
77,addis_ababa,PM2.5,9
100,johannesburg,SO₂,10
97,johannesburg,PM10,9
98,johannesburg,PM2.5,7
90,johannesburg,CO,5
94,johannesburg,O₃,5
91,johannesburg,NO,4
92,johannesburg,NOx,4
93,johannesburg,NO₂,4
95,johannesburg,PM0.3 count,1


## 3. Data Quality Assessment


This section audits data quality before cleaning, using the raw PM2.5 daily files loaded in Section 2.

### QA checks performed

1. Missing values: missing `value` records and missing coverage fields by city and year.
2. Duplicate records: repeated `(city, sensor_id, date_utc)` daily rows.
3. Outliers:
   - Physically implausible values (`value < 0` or `value > 500` µg/m³).
   - Statistical outliers using a city-level IQR rule.
4. Completeness: station-level continuity (`actual_days / expected_days`) and average OpenAQ `percentComplete`.
5. Consistency: units, averaging period labels/intervals, timezone offsets, and date ordering checks.

All QA outputs are shown as tables below and will directly inform Section 4 cleaning decisions.

In [18]:
import numpy as np
from IPython.display import display

rows = []
for city, city_stations in measurements.items():
    for station in city_stations:
        for day in station.get("daily_averages", []):
            period = day.get("period", {})
            dt_from = period.get("datetimeFrom", {})
            dt_to = period.get("datetimeTo", {})
            parameter = day.get("parameter", {})
            coverage = day.get("coverage", {})

            rows.append({
                "city": city,
                "station_id": station.get("station_id"),
                "station_name": station.get("station_name"),
                "sensor_id": station.get("sensor_id"),
                "date_utc": dt_from.get("utc"),
                "date_local": dt_from.get("local"),
                "date_to_utc": dt_to.get("utc"),
                "value": day.get("value"),
                "unit": parameter.get("units"),
                "parameter_name": parameter.get("name"),
                "period_label": period.get("label"),
                "period_interval": period.get("interval"),
                "expected_count": coverage.get("expectedCount"),
                "observed_count": coverage.get("observedCount"),
                "percent_complete": coverage.get("percentComplete"),
                "has_flags": day.get("flagInfo", {}).get("hasFlags"),
            })

qa_df = pd.DataFrame(rows)
qa_df["date_utc"] = pd.to_datetime(qa_df["date_utc"], errors="coerce", utc=True)
qa_df["date_to_utc"] = pd.to_datetime(qa_df["date_to_utc"], errors="coerce", utc=True)
qa_df["year"] = qa_df["date_utc"].dt.year

print(f"Total PM2.5 daily rows in scope: {len(qa_df):,}")
print(f"Cities covered: {qa_df['city'].nunique()} | Stations covered: {qa_df['station_id'].nunique()} | Sensors covered: {qa_df['sensor_id'].nunique()}")




Total PM2.5 daily rows in scope: 22,715
Cities covered: 6 | Stations covered: 119 | Sensors covered: 119


In [19]:
# Missing Values:
missing_city = (
    qa_df.groupby("city", as_index=False)
    .agg(
        total_rows=("value", "size"),
        missing_value_rows=("value", lambda s: s.isna().sum()),
        missing_percent_complete=("percent_complete", lambda s: s.isna().sum()),
        missing_date_rows=("date_utc", lambda s: s.isna().sum()),
    )
)
missing_city["missing_value_pct"] = (100 * missing_city["missing_value_rows"] / missing_city["total_rows"]).round(2)
missing_city["missing_percent_complete_pct"] = (
    100 * missing_city["missing_percent_complete"] / missing_city["total_rows"]
).round(2)
display(missing_city.sort_values("missing_value_pct", ascending=False))

missing_year = (
    qa_df.groupby(["city", "year"], as_index=False)
    .agg(
        total_rows=("value", "size"),
        missing_value_rows=("value", lambda s: s.isna().sum()),
    )
)
missing_year["missing_value_pct"] = (100 * missing_year["missing_value_rows"] / missing_year["total_rows"]).round(2)
display(missing_year.sort_values(["city", "year"]))

,city,total_rows,missing_value_rows,missing_percent_complete,missing_date_rows,missing_value_pct,missing_percent_complete_pct
0,addis_ababa,2851,366,0,0,12.84,0.0
2,kampala,7271,250,0,0,3.44,0.0
4,lagos,9064,17,0,0,0.19,0.0
1,johannesburg,3,0,0,0,0.00,0.0
3,kigali,1418,0,0,0,0.00,0.0
5,nairobi,2108,0,0,0,0.00,0.0


,city,year,total_rows,missing_value_rows,missing_value_pct
0,addis_ababa,2020,1,0,0.00
1,addis_ababa,2021,468,23,4.91
2,addis_ababa,2022,691,231,33.43
3,addis_ababa,2023,587,78,13.29
4,addis_ababa,2024,659,31,4.70
5,addis_ababa,2025,326,3,0.92
6,addis_ababa,2026,119,0,0.00
7,johannesburg,2026,3,0,0.00
8,kampala,2020,1,1,100.00
9,kampala,2021,356,36,10.11


In [20]:
# Identify duplicate records based on city, station_id, sensor_id, and date_utc.
dup_mask = qa_df.duplicated(subset=["city", "sensor_id", "date_utc"], keep=False)
dup_rows = qa_df[dup_mask].sort_values(["city", "sensor_id", "date_utc"] )
duplicate_summary = (
    qa_df.groupby("city", as_index=False)
    .apply(lambda g: pd.Series({
        "duplicate_rows": g.duplicated(subset=["sensor_id", "date_utc"]).sum(),
        "total_rows": len(g),
    }))
    .reset_index(drop=True)
)
duplicate_summary["duplicate_pct"] = (100 * duplicate_summary["duplicate_rows"] / duplicate_summary["total_rows"]).round(4)
display(duplicate_summary.sort_values("duplicate_rows", ascending=False))

if dup_rows.empty:
    print("No duplicate (city, sensor_id, date_utc) rows detected.")
else:
    print("Sample duplicate rows:")
    display(dup_rows.head(20))

,city,duplicate_rows,total_rows,duplicate_pct
0,addis_ababa,0,2851,0.0
1,johannesburg,0,3,0.0
2,kampala,0,7271,0.0
3,kigali,0,1418,0.0
4,lagos,0,9064,0.0
5,nairobi,0,2108,0.0


No duplicate (city, sensor_id, date_utc) rows detected.


In [21]:
# Outliers check using IQR method
qa_df["implausible_physical"] = (qa_df["value"] < 0) | (qa_df["value"] > 500)

# City-level IQR rule (computed on non-missing values).
city_q = qa_df.groupby("city")["value"].quantile([0.25, 0.75]).unstack()
city_q.columns = ["q1", "q3"]
city_q["iqr"] = city_q["q3"] - city_q["q1"]
city_q["lower_iqr"] = city_q["q1"] - 1.5 * city_q["iqr"]
city_q["upper_iqr"] = city_q["q3"] + 1.5 * city_q["iqr"]

qa_df = qa_df.merge(city_q[["lower_iqr", "upper_iqr"]], left_on="city", right_index=True, how="left")
qa_df["iqr_outlier"] = (qa_df["value"] < qa_df["lower_iqr"]) | (qa_df["value"] > qa_df["upper_iqr"] )
qa_df.loc[qa_df["value"].isna(), "iqr_outlier"] = False

outlier_summary = (
    qa_df.groupby("city", as_index=False)
    .agg(
        total_rows=("value", "size"),
        physical_outliers=("implausible_physical", "sum"),
        iqr_outliers=("iqr_outlier", "sum"),
    )
)
outlier_summary["physical_outlier_pct"] = (100 * outlier_summary["physical_outliers"] / outlier_summary["total_rows"]).round(4)
outlier_summary["iqr_outlier_pct"] = (100 * outlier_summary["iqr_outliers"] / outlier_summary["total_rows"]).round(2)
display(outlier_summary.sort_values("iqr_outlier_pct", ascending=False))


,city,total_rows,physical_outliers,iqr_outliers,physical_outlier_pct,iqr_outlier_pct
4,lagos,9064,72,584,0.7944,6.44
2,kampala,7271,33,392,0.4539,5.39
5,nairobi,2108,0,87,0.0000,4.13
0,addis_ababa,2851,0,93,0.0000,3.26
3,kigali,1418,0,33,0.0000,2.33
1,johannesburg,3,0,0,0.0000,0.00


In [22]:
# Completeness check: median coverage ratio and mean daily percent complete per city.
station_completeness = (
    qa_df.groupby(["city", "station_id", "station_name", "sensor_id"], as_index=False)
    .agg(
        first_date=("date_utc", "min"),
        last_date=("date_utc", "max"),
        actual_days=("date_utc", "nunique"),
        mean_percent_complete=("percent_complete", "mean"),
        non_missing_values=("value", lambda s: s.notna().sum()),
    )
)
station_completeness["expected_days"] = (
    (station_completeness["last_date"] - station_completeness["first_date"]).dt.days + 1
)
station_completeness["coverage_ratio"] = (
    station_completeness["actual_days"] / station_completeness["expected_days"]
).replace([np.inf, -np.inf], np.nan)

# Practical quality bands for this project.
station_completeness["quality_band"] = pd.cut(
    station_completeness["coverage_ratio"],
    bins=[-0.01, 0.5, 0.8, 0.95, 1.0],
    labels=["very low", "low", "medium", "high"],
    include_lowest=True,
 )

city_completeness = (
    station_completeness.groupby("city", as_index=False)
    .agg(
        stations=("station_id", "count"),
        median_coverage_ratio=("coverage_ratio", "median"),
        mean_daily_percent_complete=("mean_percent_complete", "mean"),
    )
)
city_completeness["median_coverage_ratio"] = city_completeness["median_coverage_ratio"].round(3)
city_completeness["mean_daily_percent_complete"] = city_completeness["mean_daily_percent_complete"].round(2)
display(city_completeness.sort_values("median_coverage_ratio", ascending=False))

print("Lowest-coverage stations (top 15):")
display(
    station_completeness.sort_values(["coverage_ratio", "mean_percent_complete"], ascending=[True, True])
    [["city", "station_name", "sensor_id", "actual_days", "expected_days", "coverage_ratio", "mean_percent_complete", "quality_band"]]
    .head(15)
)

,city,stations,median_coverage_ratio,mean_daily_percent_complete
0,addis_ababa,7,1.000,34.25
1,johannesburg,1,1.000,87.67
3,kigali,3,1.000,73.49
5,nairobi,12,0.984,81.71
2,kampala,35,0.937,48.71
4,lagos,61,0.878,53.19


Lowest-coverage stations (top 15):


,city,station_name,sensor_id,actual_days,expected_days,coverage_ratio,mean_percent_complete,quality_band
81,lagos,"Miri Air - Diamond School, Ogijo",15915968,6,152,0.039474,44.833333,very low
28,kampala,airqo_g5449,14507651,7,103,0.067961,7.142857,very low
57,lagos,Oshodi,13520831,2,17,0.117647,4.000000,very low
104,lagos,LAMATA Place,17066590,2,11,0.181818,31.000000,very low
66,lagos,Ikeja Bus Terminal,13928964,50,223,0.224215,7.320000,very low
71,lagos,Ojuelegba QBC,14010806,7,30,0.233333,31.571429,very low
51,lagos,Shagisha - Magodo,13418561,114,320,0.356250,60.964912,very low
118,nairobi,Providence Academy,15427115,46,125,0.368000,76.304348,very low
107,nairobi,Nairobi RR,2002532,371,973,0.381295,93.035040,very low
61,lagos,LAMATA Place,13557362,37,88,0.420455,18.405405,very low


In [23]:
# Consistency checks: check for any negative values in the 'value' column and any missing units or parameter names.

units_check = qa_df.groupby("city", as_index=False)["unit"].nunique().rename(columns={"unit": "unique_units"})
period_label_check = qa_df.groupby("city", as_index=False)["period_label"].nunique().rename(columns={"period_label": "unique_period_labels"})
period_interval_check = qa_df.groupby("city", as_index=False)["period_interval"].nunique().rename(columns={"period_interval": "unique_period_intervals"})

consistency_summary = units_check.merge(period_label_check, on="city").merge(period_interval_check, on="city")
display(consistency_summary)

print("Unit values observed:")
display(
    qa_df[["city", "unit"]].drop_duplicates().sort_values(["city", "unit"]).reset_index(drop=True)
)

print("Period labels and intervals observed:")
display(
    qa_df[["city", "period_label", "period_interval"]]
    .drop_duplicates()
    .sort_values(["city", "period_label", "period_interval"] )
    .reset_index(drop=True)
)

date_order_issues = qa_df[qa_df["date_to_utc"] < qa_df["date_utc"]]
print(f"Rows with date_to_utc earlier than date_utc: {len(date_order_issues)}")

,city,unique_units,unique_period_labels,unique_period_intervals
0,addis_ababa,1,1,1
1,johannesburg,1,1,1
2,kampala,1,1,1
3,kigali,1,1,1
4,lagos,1,1,1
5,nairobi,1,1,1


Unit values observed:


,city,unit
0,addis_ababa,µg/m³
1,johannesburg,µg/m³
2,kampala,µg/m³
3,kigali,µg/m³
4,lagos,µg/m³
5,nairobi,µg/m³


Period labels and intervals observed:


,city,period_label,period_interval
0,addis_ababa,1day,24:00:00
1,johannesburg,1day,24:00:00
2,kampala,1day,24:00:00
3,kigali,1day,24:00:00
4,lagos,1day,24:00:00
5,nairobi,1day,24:00:00


Rows with date_to_utc earlier than date_utc: 0


In [25]:
# Conclusion: The QA checks reveal that there are missing values, duplicate records, outliers, and completeness issues in the dataset.

qa_conclusion = missing_city[["city", "missing_value_pct"]].merge(
    duplicate_summary[["city", "duplicate_rows"]], on="city", how="left"
).merge(
    outlier_summary[["city", "physical_outliers", "iqr_outlier_pct"]], on="city", how="left"
).merge(
    city_completeness[["city", "median_coverage_ratio", "mean_daily_percent_complete"]], on="city", how="left"
)
display(qa_conclusion.sort_values(["missing_value_pct", "median_coverage_ratio"], ascending=[False, True]))


,city,missing_value_pct,duplicate_rows,physical_outliers,iqr_outlier_pct,median_coverage_ratio,mean_daily_percent_complete
0,addis_ababa,12.84,0,0,3.26,1.000,34.25
2,kampala,3.44,0,33,5.39,0.937,48.71
4,lagos,0.19,0,72,6.44,0.878,53.19
5,nairobi,0.00,0,0,4.13,0.984,81.71
1,johannesburg,0.00,0,0,0.00,1.000,87.67
3,kigali,0.00,0,0,2.33,1.000,73.49


## 4. Cleaning & Preprocessing

**TODO:** Handle missing values, de-duplicate, standardize timestamps/timezones/units, merge per-city measurement data into one master DataFrame with a `city` column. Document every decision made. Output: data/cleaned/air_quality_master.csv

In [ ]:
# TODO: 4. Cleaning & Preprocessing


## 5. Exploratory Data Analysis & Statistical Techniques

**Owner:** Winnie

**TODO:** Apply at least 3 statistical techniques answering Q1, Q3, Q4, Q5: descriptive/ranking (highest avg PM2.5), trend/seasonal analysis, WHO exceedance-rate analysis, completeness scoring.

In [ ]:
# TODO (Winnie): 5. Exploratory Data Analysis & Statistical Techniques


## 6. Visualizations

**Owner:** Gloria

**TODO:** At least 5 charts, each with a one-line justification for the chart type chosen. Suggested: city PM2.5 ranking, pollutant frequency by city, seasonal variation, WHO exceedance over time, station completeness, station location map.

In [ ]:
# TODO (Gloria): 6. Visualizations


## 7. Findings & Recommendations

**Owner:** Gloria (with input from all)

**TODO:** Evidence-backed answers to all 6 client questions, each citing the specific stat/chart that supports it. Prioritization recommendation for UNEP: which cities need intervention, and why.

In [ ]:
# TODO (Gloria (with input from all)): 7. Findings & Recommendations
